# 05 · SQLite + Chroma 적재 데이터 검증

**목적**: 이미 적재된 SQLite(`data/eddr.sqlite`)와 Chroma(`data/index/chroma`)의 데이터가
설계 명세(ADR·PLAN)대로 들어갔고, **서비스 단계(⑦ tool surface)를 올릴 수 있는 수준인지**를
**raw 접근**으로 검증한다. `src/eddr`는 import하지 않는다.

- **§A** 스키마·필드 정의 vs 설계 명세 대조
- **§B** SQLite ↔ Chroma 정합성 (count · id 집합 · photo_id 무결성 · 차원 · **캡션 본문 전수 일치**)
- **§C** 저장 데이터 상세 검수 — 파일 실존 · 필드 충전율 · 캡션 품질 · **레코드 추적 + 실제 사진↔캡션 육안 대응**
- **§D** 검색 동작 검증 — 한국어 쿼리 top-5를 **실제 사진으로 표출** + self-retrieval 정량 게이트
- **§E** 종합 리포트 + 과거 불일치 해소 확인

**실행**: 커널 `eddr-eda` (= .venv). **읽기 전용** — DB/Chroma/코드를 변경하지 않는다(SELECT · PRAGMA · get · query만).

**privacy**: 출력에 실제 사진·정밀좌표가 포함되므로 이 노트북은 **git 미추적**(`.gitignore`의 `notebooks/*.ipynb`)을 전제로 한다.
ADR-0001 경계는 "외부 LLM 전송 금지"이며 로컬 표시와는 무관.

In [ ]:
# 환경 준비 — 경로 상수, 연결, 이미지 헬퍼, 사전 점검 (모두 읽기 전용)
import re
import sqlite3
import textwrap
from pathlib import Path

import chromadb
import matplotlib.pyplot as plt
import numpy as np
import ollama
import pandas as pd
from IPython.display import display
from PIL import Image, ImageOps
from pillow_heif import register_heif_opener

register_heif_opener()  # HEIC(최다 포맷)을 PIL로 열기 위한 플러그인
plt.rcParams["font.family"] = "AppleGothic"  # 한국어 쿼리·라벨용 (macOS)
plt.rcParams["axes.unicode_minus"] = False

# 노트북이 notebooks/ 또는 루트 어디서 실행돼도 ROOT를 찾는다 (기존 01~04 패턴)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = ROOT / "data" / "eddr.sqlite"
CHROMA_PATH = ROOT / "data" / "index" / "chroma"
COLLECTION = "eddr_caption_text_v1"
EMBED_MODEL = "qwen3-embedding:8b"
SEED = 42  # 표본 재현성 — 재실행해도 같은 사진이 뽑혀 비교 가능

assert DB_PATH.exists(), f"SQLite 없음: {DB_PATH}"
assert CHROMA_PATH.exists(), f"Chroma 디렉터리 없음: {CHROMA_PATH}"

# 읽기 전용 SQLite 연결 (URI mode=ro 로 쓰기 자체를 차단)
conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
conn.row_factory = sqlite3.Row


def q(sql: str, params: tuple = ()) -> pd.DataFrame:
    """SELECT/PRAGMA 결과를 DataFrame으로 반환하는 헬퍼."""
    return pd.read_sql_query(sql, conn, params=params)


def resolve_path(p: str) -> Path:
    """DB image_path 해석 — 절대경로는 그대로, 상대경로는 레포 루트(ROOT) 기준.

    google_takeout만 상대경로로 적재돼 있어(§C-1 실측) CWD 의존을 끊으려면 이 규칙이 필요하다.
    """
    path = Path(p)
    return path if path.is_absolute() else ROOT / path


def load_thumb(path: str, max_px: int = 512):
    """이미지를 썸네일 PIL Image로 로드. RAW(rw2·orf 등) 미지원 포맷·미존재 파일은 None."""
    try:
        img = Image.open(resolve_path(path))
        img = ImageOps.exif_transpose(img)  # EXIF 회전 보정 (세로 사진이 눕지 않게)
        img.thumbnail((max_px, max_px))
        return img.convert("RGB")
    except Exception:
        return None


def show_photo_grid(items: list[dict], title: str = "", cols: int = 5, cell_in: float = 2.5):
    """{'path', 'label'} 목록을 썸네일 그리드로 표출. 미지원 포맷은 회색 placeholder."""
    rows = -(-len(items) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * cell_in, rows * (cell_in + 0.9)))
    axes = np.atleast_1d(axes).ravel()
    for ax, item in zip(axes, items):
        img = load_thumb(item["path"])
        if img is not None:
            ax.imshow(img)
        else:
            ax.set_facecolor("0.85")
            ax.text(0.5, 0.5, f"미리보기 불가\n({Path(item['path']).suffix})",
                    ha="center", va="center", fontsize=8, transform=ax.transAxes)
        ax.set_title(item["label"], fontsize=7)
        ax.axis("off")
    for ax in axes[len(items):]:
        ax.axis("off")
    if title:
        fig.suptitle(title, fontsize=11)
        fig.tight_layout(rect=(0, 0, 1, 0.94))
    else:
        fig.tight_layout()
    plt.show()


# Chroma 컬렉션 열기 (embedding_function 미지정 — 적재 때와 동일)
client = chromadb.PersistentClient(path=str(CHROMA_PATH))
col = client.get_collection(COLLECTION)

# 쿼리 임베딩 모델 가용성 확인
installed = {m.model for m in ollama.list().models}
assert EMBED_MODEL in installed, f"{EMBED_MODEL} 미설치. 설치된 모델: {sorted(installed)}"

print(f"ROOT        = {ROOT}")
print(f"SQLite      = {DB_PATH.name} ({DB_PATH.stat().st_size / 1e6:.1f} MB)")
print(f"Chroma      = {CHROMA_PATH}")
print(f"Collection  = {COLLECTION}  (count={col.count():,})")
print(f"Embed model = {EMBED_MODEL}  (installed: OK)")

## §A — 스키마·필드 정의 vs 설계 명세 대조

설계 명세(`docs/PLAN.md`, ADR, `repository.py`)가 규정한 컬럼·제약·도메인 값과
실제 DB를 대조한다.

In [ ]:
# §A-1: 실제 테이블 컬럼을 설계 명세(기대값)와 대조
EXPECTED_COLUMNS = {
    "photos": ["id", "source", "source_uri", "image_path", "content_hash",
               "perceptual_hash", "taken_at", "latitude", "longitude", "width",
               "height", "camera_make", "camera_model", "indexing_status",
               "created_at", "updated_at"],
    "captions": ["photo_id", "model_id", "lang", "text", "generated_at"],
    "embeddings": ["photo_id", "kind", "model_id", "vector_id", "dimensions", "created_at"],
    "index_errors": ["photo_id", "stage", "message", "created_at"],
}

rows = []
for tbl, expected in EXPECTED_COLUMNS.items():
    actual = list(q(f"PRAGMA table_info({tbl})")["name"])
    missing = [c for c in expected if c not in actual]
    extra = [c for c in actual if c not in expected]
    rows.append({
        "table": tbl,
        "expected": len(expected),
        "actual": len(actual),
        "missing": ", ".join(missing) or "—",
        "extra": ", ".join(extra) or "—",
        "verdict": "PASS" if not missing and not extra else "MISMATCH",
    })
schema_check = pd.DataFrame(rows)
schema_check

In [ ]:
# §A-2: photos 컬럼 상세 — 타입·NOT NULL·PK (privacy 필드 latitude/longitude 포함 확인)
q("PRAGMA table_info(photos)")[["name", "type", "notnull", "pk"]]

In [ ]:
# §A-3: 인덱스·UNIQUE 제약 (설계: idx_photos_status, idx_photos_content_hash, UNIQUE(source, source_uri))
idx = q("PRAGMA index_list(photos)")
display(idx[["name", "unique", "origin"]])
for name in idx["name"]:
    members = q(f"PRAGMA index_info({name})")["name"].tolist()
    print(f"{name}: {members}")

In [ ]:
# §A-4: 도메인(enum) 값 실측 vs 설계 명세
#   기대값 출처(SSoT): src/eddr/db/repository.py VALID_SOURCES·INDEXING_STATUSES
#   (doc-contract hook이 PLAN.md §4·wiki/architecture/db-schema.md와 동기화를 강제)
#   노트북 원칙상 import하지 않고 리터럴로 대조한다 — drift 시 §E-2가 잡는다.
DESIGN_SOURCES = {"photos_library", "google_takeout", "local"}
DESIGN_STATUS = {"meta_done", "missing_image", "caption_done", "skipped_video", "trip_assigned"}

actual_sources = set(q("SELECT DISTINCT source FROM photos")["source"])
actual_status = set(q("SELECT DISTINCT indexing_status FROM photos")["indexing_status"])

print("photos.source        :", sorted(actual_sources))
print("\nphotos.indexing_status:")
display(q("SELECT indexing_status, COUNT(*) AS n FROM photos GROUP BY 1 ORDER BY 2 DESC"))
print("captions (model_id, lang):")
display(q("SELECT model_id, lang, COUNT(*) AS n FROM captions GROUP BY 1, 2 ORDER BY 3 DESC"))
print("embeddings (kind, model_id, dimensions):")
display(q("SELECT kind, model_id, dimensions, COUNT(*) AS n FROM embeddings GROUP BY 1, 2, 3 ORDER BY 4 DESC"))

enum_ok = actual_sources <= DESIGN_SOURCES and actual_status <= DESIGN_STATUS
print("\n설계 외 source         :", (actual_sources - DESIGN_SOURCES) or "없음")
print("설계 외 indexing_status:", (actual_status - DESIGN_STATUS) or "없음")
print("enum 검증              :", "PASS" if enum_ok else "MISMATCH")

In [ ]:
# §A-5: vector_id 형식 검증 — 'caption_text:<photo_id>:qwen3-embedding:8b'
#   주의: photo_id 자체가 'photos_library:<uuid>'처럼 콜론을 포함하므로 split(':') 금지.
#         prefix/suffix 패턴 + 재구성 비교로만 검증한다.
emb = q("SELECT photo_id, kind, model_id, vector_id, dimensions FROM embeddings")
pat = re.compile(r"^caption_text:.+:qwen3-embedding:8b$")
emb["vid_ok"] = emb["vector_id"].map(lambda v: bool(pat.match(v)))
emb["vid_reconstructs"] = emb.apply(
    lambda r: r["vector_id"] == f"{r['kind']}:{r['photo_id']}:{r['model_id']}", axis=1
)
print(f"vector_id 패턴 매칭   : {emb['vid_ok'].mean() * 100:.1f}%  ({emb['vid_ok'].sum():,}/{len(emb):,})")
print(f"vector_id 재구성 일치 : {emb['vid_reconstructs'].mean() * 100:.1f}%")
print(f"dimensions 고유값     : {sorted(emb['dimensions'].unique())}")
emb.head(3)

## §B — SQLite ↔ Chroma 정합성

SQLite `embeddings`(canonical 장부)와 Chroma 벡터 스토어가 1:1로 맞물리는지 확인한다.

In [ ]:
# §B-1: count 비교 (SQLite embeddings == Chroma)
n_sql = int(q("SELECT COUNT(*) AS n FROM embeddings")["n"].iloc[0])
n_chroma = col.count()
print(f"SQLite embeddings rows : {n_sql:,}")
print(f"Chroma vectors         : {n_chroma:,}")
print(f"일치 여부              : {'PASS' if n_sql == n_chroma else 'MISMATCH'}")

In [ ]:
# §B-2: id 집합 양방향 diff — SQLite vector_id 집합 vs Chroma id 집합
sql_ids = set(q("SELECT vector_id FROM embeddings")["vector_id"])
chroma_ids = set(col.get(include=[])["ids"])  # ids는 include와 무관하게 항상 반환
only_sql = sql_ids - chroma_ids
only_chroma = chroma_ids - sql_ids
print(f"SQLite vector_id        : {len(sql_ids):,}")
print(f"Chroma id               : {len(chroma_ids):,}")
print(f"SQLite에만 (Chroma 누락): {len(only_sql):,}")
print(f"Chroma에만 (고아 벡터)  : {len(only_chroma):,}")
if only_sql:
    print("  예시(SQLite만):", list(only_sql)[:3])
if only_chroma:
    print("  예시(Chroma만):", list(only_chroma)[:3])

In [ ]:
# §B-3: photo_id 외래 무결성 — caption/embedding의 photo_id가 photos에 존재하는가
orphan_emb = int(q("""
    SELECT COUNT(*) AS n FROM embeddings e
    LEFT JOIN photos p ON e.photo_id = p.id
    WHERE p.id IS NULL
""")["n"].iloc[0])
orphan_cap = int(q("""
    SELECT COUNT(*) AS n FROM captions c
    LEFT JOIN photos p ON c.photo_id = p.id
    WHERE p.id IS NULL
""")["n"].iloc[0])
cap_no_emb = int(q("""
    SELECT COUNT(DISTINCT c.photo_id) AS n FROM captions c
    LEFT JOIN embeddings e ON c.photo_id = e.photo_id
    WHERE e.photo_id IS NULL
""")["n"].iloc[0])
print(f"고아 embedding (photos 없음): {orphan_emb:,}")
print(f"고아 caption  (photos 없음): {orphan_cap:,}")
print(f"caption 있고 embedding 없음 : {cap_no_emb:,}")

In [ ]:
# §B-4: Chroma metadata 필드 완전성 + 실제 벡터 차원 + source 교차검증
EXPECTED_META = {"photo_id", "source", "kind", "model_id"}
sample = col.get(limit=200, include=["metadatas", "embeddings", "documents"])

meta_keys = set()
for m in sample["metadatas"]:
    meta_keys |= set(m.keys())
print(f"metadata 키(샘플 200): {sorted(meta_keys)}")
print(f"기대 키 누락         : {(EXPECTED_META - meta_keys) or '없음'}")

vec_dim = len(sample["embeddings"][0])
print(f"Chroma 벡터 차원     : {vec_dim}  (SQLite dimensions: {sorted(emb['dimensions'].unique())})")

# metadata.source가 SQLite photos.source와 일치하는지 (샘플 교차검증)
src_map = dict(q("SELECT id, source FROM photos").itertuples(index=False, name=None))
mismatch = sum(
    1 for m in sample["metadatas"]
    if m.get("photo_id") in src_map and m.get("source") != src_map[m["photo_id"]]
)
print(f"source 불일치(샘플)  : {mismatch}")

In [ ]:
# §B-5: 캡션 본문 전수 일치 — Chroma documents == SQLite captions.text (9,383건 전부)
#   id·count 정합(§B-1~2)만으로는 "벡터에 달린 본문이 그 사진의 캡션"임을 보장하지 못한다.
cap_df = q("SELECT photo_id, text FROM captions")
chroma_all = col.get(include=["documents", "metadatas"])
doc_by_pid = {m["photo_id"]: d for m, d in zip(chroma_all["metadatas"], chroma_all["documents"])}

cap_df["doc_match"] = [
    doc_by_pid.get(pid) == txt for pid, txt in zip(cap_df["photo_id"], cap_df["text"])
]
n_doc_mismatch = int((~cap_df["doc_match"]).sum())
print(f"captions 행          : {len(cap_df):,}")
print(f"Chroma documents 보유: {len(doc_by_pid):,}")
print(f"본문 불일치          : {n_doc_mismatch:,}건  → {'PASS' if n_doc_mismatch == 0 else 'FAIL'}")
if n_doc_mismatch:
    display(cap_df[~cap_df["doc_match"]].head())

## §C — 저장 데이터 상세 검수 (서비스 가능성 증거)

집계·정합(§A·§B)만으로는 **내용이 올바른가**를 입증할 수 없다. 여기서는 개별 레코드 수준으로 내려간다:

1. **§C-1** 이미지 파일 실존 전수 검사 + 포맷별 썸네일 로드 가능성 실측
2. **§C-2** 소스별 메타 필드 충전율 (서비스에서 믿고 쓸 수 있는 필드 식별)
3. **§C-3** 캡션 텍스트 품질 — 길이 분포 · `Search keywords:` 구조 커버리지
4. **§C-4** 레코드 추적 — 사진 1장의 4계층(photos→captions→embeddings→Chroma) 저장 상태를 원본 그대로
5. **§C-5** 무작위 9장의 **실제 사진과 캡션을 나란히 표출** → 사람이 대응을 직접 판정

In [ ]:
# §C-1: 경로 표기 실태 + 이미지 파일 실존 전수 검사 + 포맷별 썸네일 로드 실측
#   발견(2026-06-10 1차 실행): google_takeout 1,385건만 상대경로로 적재됨(타 소스는 절대경로).
#   서비스 코드가 CWD에 의존하면 깨지는 잠재 결함 → 표기 통일은 load-sources 수정(TODO ⑤ 후속)과 묶음.
#   존재 검사는 resolve_path(절대 그대로·상대는 ROOT 기준)로 해석해 수행한다.
ph = q("SELECT id, source, image_path, indexing_status, taken_at FROM photos")
ph["ext"] = ph["image_path"].str.rsplit(".", n=1).str[-1].str.lower()
cap_ph = ph[ph["indexing_status"] == "caption_done"].copy()
print(f"caption_done 사진 : {len(cap_ph):,}   (image_path NULL: {int(cap_ph['image_path'].isna().sum()):,})")

# 경로 표기 실태 — 절대 vs 상대 혼재
cap_ph["is_abs"] = cap_ph["image_path"].str.startswith("/")
style = cap_ph.groupby("source").agg(n=("id", "size"), 절대경로=("is_abs", "sum"))
style["상대경로"] = style["n"] - style["절대경로"]
display(style)
path_mixed = cap_ph["is_abs"].nunique() > 1
print("경로 표기 혼재    :", "있음 — 상대경로는 레포 루트 기준으로 해석함(데이터 결함 아님, 표기 비일관)" if path_mixed else "없음")

# 파일 실존 전수 검사 (9,383건 stat)
cap_ph["file_exists"] = [resolve_path(p).exists() for p in cap_ph["image_path"]]
n_missing_file = int((~cap_ph["file_exists"]).sum())
print(f"파일 미존재       : {n_missing_file:,}건  → {'PASS' if n_missing_file == 0 else 'FAIL'}")
if n_missing_file:
    display(cap_ph[~cap_ph["file_exists"]][["id", "source", "image_path"]].head())

# 포맷별: 건수·실존 + 대표 1건 썸네일 로드 실측 (가정이 아니라 직접 열어본다)
fmt_rows = []
for ext, grp in cap_ph.groupby("ext"):
    ok = load_thumb(grp["image_path"].iloc[0], max_px=64) is not None
    fmt_rows.append({"ext": ext, "n": len(grp), "files_exist": int(grp["file_exists"].sum()),
                     "썸네일_로드": "OK" if ok else "불가 → placeholder"})
display(pd.DataFrame(fmt_rows).sort_values("n", ascending=False).reset_index(drop=True))

In [ ]:
# §C-2: 소스별 메타 필드 충전율 — 어떤 필드를 서비스에서 믿고 쓸 수 있는가
fill = q("""
    SELECT source,
           COUNT(*)                                       AS n,
           ROUND(AVG(taken_at     IS NOT NULL) * 100, 1)  AS taken_at_pct,
           ROUND(AVG(latitude     IS NOT NULL) * 100, 1)  AS gps_pct,
           ROUND(AVG(width        IS NOT NULL) * 100, 1)  AS size_pct,
           ROUND(AVG(camera_make  IS NOT NULL) * 100, 1)  AS camera_pct,
           ROUND(AVG(content_hash IS NOT NULL) * 100, 1)  AS hash_pct
    FROM photos
    WHERE indexing_status = 'caption_done'
    GROUP BY source ORDER BY n DESC
""")
display(fill)
print("주: GPS·camera 결측은 원본 EXIF 부재로 정상 범주 — 검색은 캡션·시각 축으로도 동작해야 함(§D).")

# taken_at 포맷 유효성 — 채워져 있어도 SQLite date()로 파싱 불가면 날짜 축에서 조용히 누락된다
fmt_bad = q("""
    SELECT source,
           SUM(taken_at IS NOT NULL)                            AS has_taken_at,
           SUM(taken_at IS NOT NULL AND date(taken_at) IS NULL) AS unparsable
    FROM photos WHERE indexing_status = 'caption_done'
    GROUP BY source ORDER BY source
""")
display(fmt_bad)
n_unparsable = int(fmt_bad["unparsable"].sum())
if n_unparsable:
    sample_bad = q("SELECT taken_at FROM photos WHERE taken_at IS NOT NULL AND date(taken_at) IS NULL LIMIT 1")
    print(f"⚠️ 파싱 불가 taken_at {n_unparsable:,}건 — 예: '{sample_bad['taken_at'].iloc[0]}'"
          " (EXIF 콜론 포맷). 날짜 질의·Trip 클러스터링(⑥)에서 누락됨 → 로더 정규화 필요(TODO).")
else:
    print("taken_at 전건 SQLite date() 파싱 가능.")

In [ ]:
# §C-3: 캡션 텍스트 품질 — 길이 분포 + 'Search keywords:' 구조 커버리지
texts = cap_df["text"]
lens = texts.str.len()
print(f"캡션 길이(문자): min {lens.min()} / median {int(lens.median())} / mean {lens.mean():.0f} / max {lens.max()}")
print(f"빈 캡션        : {int((lens == 0).sum())}건")

# vision 프롬프트가 요구한 'Search keywords:' 줄 — bold/plain 두 표기 혼재 (⑦ 파싱 시 둘 다 처리, TODO 반영됨)
kw_bold = texts.str.contains(r"\*\*Search keywords:\*\*", regex=True)
kw_plain = texts.str.contains("Search keywords:", regex=False) & ~kw_bold
kw_cov = (kw_bold | kw_plain).mean()
print(f"keywords 라인  : bold {int(kw_bold.sum()):,} + plain {int(kw_plain.sum()):,}"
      f" = {int((kw_bold | kw_plain).sum()):,} / {len(texts):,} ({kw_cov * 100:.1f}%)")

fig, ax = plt.subplots(figsize=(7, 2.5))
ax.hist(lens, bins=60)
ax.set_xlabel("캡션 길이(문자)")
ax.set_ylabel("사진 수")
ax.set_title("캡션 길이 분포")
plt.tight_layout()
plt.show()

In [ ]:
# §C-4: 레코드 추적 — 사진 1장이 photos → captions → embeddings → Chroma 4계층에 어떻게 저장되는가
#   소스별 1장(seed 고정)을 계층별 원본 그대로 출력하고 실제 사진까지 표시한다.
trace_ids = [grp.sample(1, random_state=SEED)["id"].iloc[0] for _, grp in cap_ph.groupby("source")]

for pid in trace_ids:
    print("=" * 100)
    photo = q("SELECT * FROM photos WHERE id = ?", (pid,)).iloc[0]
    print(f"[1. photos]  {pid}")
    display(photo.to_frame("값"))  # 전 컬럼 — 좌표 포함(로컬 표시 전용, ADR-0001은 외부 LLM 전송 경계)

    cap = q("SELECT * FROM captions WHERE photo_id = ?", (pid,)).iloc[0]
    print(f"[2. captions]  model={cap['model_id']}  lang={cap['lang']}  generated_at={cap['generated_at']}")
    print(textwrap.indent(cap["text"], "    "))

    emb_row = q("SELECT * FROM embeddings WHERE photo_id = ?", (pid,)).iloc[0]
    print(f"\n[3. embeddings]  kind={emb_row['kind']}  model={emb_row['model_id']}  dim={emb_row['dimensions']}")
    print(f"    vector_id = {emb_row['vector_id']}")

    rec = col.get(ids=[emb_row["vector_id"]], include=["embeddings", "documents", "metadatas"])
    vec = np.asarray(rec["embeddings"][0])
    print(f"[4. Chroma]  metadata = {rec['metadatas'][0]}")
    print(f"    document == captions.text : {rec['documents'][0] == cap['text']}")
    print(f"    vector dim={len(vec)}  head={np.round(vec[:6], 4).tolist()}  L2 norm={np.linalg.norm(vec):.4f}")
    print("    (CONTEXT.md 전제 검증: qwen3-embedding 정규화 벡터 → norm≈1이면 L2·cosine 순위 동일)")

    show_photo_grid(
        [{"path": photo["image_path"],
          "label": f"{photo['source']} · {str(photo['taken_at'])[:10]}"}],
        cols=1, cell_in=3.5,
    )

In [ ]:
# §C-5: 무작위 표본 — 실제 사진 ↔ 캡션 육안 대응 (소스별 3장 × 3소스, seed 고정)
#   그리드 번호와 아래 캡션 전문 번호가 1:1 대응한다. 캡션이 사진을 옳게 묘사하는지 직접 판정.
sample9 = pd.concat([
    grp.sample(min(3, len(grp)), random_state=SEED) for _, grp in cap_ph.groupby("source")
]).reset_index(drop=True)
cap_by_pid = dict(zip(cap_df["photo_id"], cap_df["text"]))

items = []
for i, r in sample9.iterrows():
    head = textwrap.shorten(cap_by_pid[r["id"]], width=120, placeholder="…")
    items.append({
        "path": r["image_path"],
        "label": f"[{i + 1}] {r['source']} · {str(r['taken_at'])[:10]}\n"
                 + "\n".join(textwrap.wrap(head, 38)[:3]),
    })
show_photo_grid(items, title="무작위 9장 — 사진과 캡션이 서로 맞는지 직접 확인", cols=3, cell_in=3.0)

for i, r in sample9.iterrows():
    print(f"\n[{i + 1}] {r['id']}  ({r['source']}, {str(r['taken_at'])[:19]})")
    print(textwrap.indent(cap_by_pid[r["id"]], "    "))

## §D — 검색 동작 검증 (실제 사진 표출)

캡션은 영어지만 임베딩 모델(`qwen3-embedding:8b`)이 multilingual이라 한국어 질의로도
검색돼야 한다. **반드시 `query_embeddings`로 검색** (컬렉션에 embedding_function이 없어
`query_texts`를 쓰면 다른 벡터공간으로 검색돼 깨진다).

- **§D-2** 한국어 키워드 쿼리 top-5를 **실제 사진 그리드로 표출** → 쿼리-사진 의미 일치를 사람이 직접 판정
- **§D-3** self-retrieval 라운드트립 — 저장된 캡션 원문으로 검색하면 그 사진이 상위에 나와야 한다
  (저장 본문 → 임베딩 → Chroma 벡터가 같은 사진으로 닫히는지에 대한 **정량 게이트**)
- **§D-4** 음성 대조 — 보유 가능성이 낮은 개념의 최근접 거리 관찰 (⑦ 거리 임계값 정책의 입력)
- **§D-5** **'어디더라?' 실사용 자연어 질의 평가** — 골든셋 후보 풀의 문장형 질문(기간·여행·지명)으로
  semantic leg 단독의 능력 지도를 그린다. 여행 폴더명·연도를 GT proxy로 자동 채점 (게이트 아님)

In [ ]:
# §D-1: 쿼리 임베딩 함수 (ollama 직접) + 차원 일치 확인
def embed_query(text: str) -> list[float]:
    return ollama.embed(model=EMBED_MODEL, input=[text]).embeddings[0]


_probe = embed_query("테스트")
print(f"쿼리 임베딩 차원: {len(_probe)}  (Chroma 차원 {vec_dim}과 일치: {len(_probe) == vec_dim})")

In [ ]:
# §D-2: 한국어 쿼리 → top-5를 실제 사진으로 표출 (rank·거리·시각·캡션 함께)
#   각 그리드에서 쿼리와 사진의 의미 일치를 직접 확인한다. 거리는 L2(제곱) — 작을수록 가깝다.
QUERIES = ["바다 풍경", "생일 케이크", "강아지", "단풍 가을", "도시 야경"]
path_by_pid = dict(zip(ph["id"], ph["image_path"]))
taken_by_pid = dict(zip(ph["id"], ph["taken_at"]))


def search(query: str, k: int = 5) -> pd.DataFrame:
    res = col.query(
        query_embeddings=[embed_query(query)],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )
    rows = []
    for rank, (doc, meta, dist) in enumerate(
        zip(res["documents"][0], res["metadatas"][0], res["distances"][0]), start=1
    ):
        pid = meta["photo_id"]
        rows.append({
            "rank": rank, "distance": round(float(dist), 3), "photo_id": pid,
            "source": meta.get("source"), "taken_at": taken_by_pid.get(pid),
            "image_path": path_by_pid.get(pid), "caption": doc,
        })
    return pd.DataFrame(rows)


search_results = {}
for query in QUERIES:
    df = search(query)
    search_results[query] = df
    items = [{
        "path": r["image_path"],
        "label": f"#{r['rank']}  d={r['distance']:.2f} · {r['source']} · {str(r['taken_at'])[:10]}\n"
                 + "\n".join(textwrap.wrap(textwrap.shorten(r["caption"], 110, placeholder="…"), 34)[:3]),
    } for _, r in df.iterrows()]
    show_photo_grid(items, title=f"쿼리 '{query}' — top-5", cols=5, cell_in=2.4)
    display(df[["rank", "distance", "source", "taken_at"]])

In [ ]:
# §D-3: self-retrieval 라운드트립 — 저장 캡션 원문으로 검색하면 그 사진이 나와야 한다
#   저장 본문 → (재)임베딩 → Chroma 최근접이 자기 자신으로 닫히는지 = 파이프라인 전체의 정량 검증.
#   연사(burst)처럼 캡션이 사실상 동일한 사진은 top-1 동률이 가능하므로 게이트는 top-5 기준.
N_SELF = 50
sample_caps = cap_df.sample(N_SELF, random_state=SEED)

hits1 = hits5 = 0
self_dists = []
for r in sample_caps.itertuples():
    res = col.query(query_embeddings=[embed_query(r.text)], n_results=5,
                    include=["metadatas", "distances"])
    pids = [m["photo_id"] for m in res["metadatas"][0]]
    hits1 += pids[0] == r.photo_id
    if r.photo_id in pids:
        hits5 += 1
        self_dists.append(res["distances"][0][pids.index(r.photo_id)])

top1_rate, top5_rate = hits1 / N_SELF, hits5 / N_SELF
self_ok = top5_rate >= 0.95
print(f"표본 {N_SELF}건 self-retrieval (seed={SEED})")
print(f"  top-1 자기 일치 : {hits1}/{N_SELF} ({top1_rate:.0%})")
print(f"  top-5 자기 포함 : {hits5}/{N_SELF} ({top5_rate:.0%})  → 게이트(≥95%): {'PASS' if self_ok else 'FAIL'}")
if self_dists:
    print(f"  자기 벡터까지 거리: median {np.median(self_dists):.4f}  (재임베딩 결정성 — 0에 가까울수록 좋음)")

In [ ]:
# §D-4: 음성 대조(관찰) — 보유 가능성이 낮은 개념은 최근접 거리가 눈에 띄게 멀어야 한다
#   PASS/FAIL 게이트가 아니다. 거리 절대값 분포는 ⑦의 거리 임계값 정책 결정에 쓰는 입력.
NEG_QUERY = "북극 오로라"
neg = search(NEG_QUERY)

pos_best = {qq: float(df["distance"].min()) for qq, df in search_results.items()}
print("긍정 쿼리 최근접 거리:", {k: round(v, 3) for k, v in pos_best.items()})
print(f"음성 쿼리 '{NEG_QUERY}' 최근접 거리: {neg['distance'].min():.3f}")
print("→ 음성 쿼리의 최근접이 긍정 쿼리들보다 충분히 멀면, 거리값이 관련도 신호로 쓸 만하다는 근거.")

items = [{
    "path": r["image_path"],
    "label": f"#{r['rank']}  d={r['distance']:.2f} · {str(r['taken_at'])[:10]}",
} for _, r in neg.iterrows()]
show_photo_grid(items, title=f"음성 대조 '{NEG_QUERY}' — 가장 가까운 5장 (실제로 관련 사진인지 눈으로 확인)",
                cols=5, cell_in=2.4)

### §D-5 — '어디더라?' 실사용 자연어 질의 평가

§D-2의 키워드 쿼리는 임베딩 동작 확인이지, **EDDR("어디더라?")이 실제로 받을 질문**에 대한
증거가 아니다. 여기서는 골든셋 후보 풀의 문장형 질문(특정 기간 여행·특정 장소·이벤트)을 그대로
던져 **semantic leg 단독**이 어디까지 답하는지 측정한다.

- **GT proxy 자동 채점**: local_photos의 여행 폴더명(`2019_이탈리아`·`2022_아이슬란드`·`200620_23_제주` 등)과
  `taken_at` 연도가 질문의 "여행/기간 스코프" 정답을 제공 → top-10 중 스코프 적합 수(hit@10)를 센다.
- **기대치(후보 풀 선별 원칙과 동일)**: 여행 고유 장면(오로라·은하수·돌로미티)은 캡션만으로도 적중 가능.
  **지명·연도 축은 캡션에 없으므로 실패가 정상** — 그 실패의 정량화가 ⑦에서 trip/geocode/날짜 필터를
  합성해야 하는 실측 근거가 된다. 따라서 이 절은 PASS/FAIL 게이트가 아니라 능력 지도다.

In [ ]:
# §D-5: '어디더라?' 실사용 자연어 질의 평가 — semantic leg 단독의 능력 지도 (게이트 아님)
#   질의는 골든셋 후보 풀(docs/GOLDEN_QUERY_CANDIDATES.md)에서 발췌·변형한 문장형.
#   GT proxy: local_photos 여행 폴더명(예: 2019_이탈리아 582장)·taken_at 연도.
#   ⚠️ 한글 폴더명은 NFD — NFC 정규화 후 비교(프로젝트 기지 사항). 골든셋 확정은 사용자 전용.
import unicodedata


def _nfc(s: str) -> str:
    return unicodedata.normalize("NFC", s) if s else ""


EDDR_QUERIES = [
    {"q": "2019년 이탈리아 여행에서 산악 풍경 찍은 사진 어디더라?", "type": "여행+장면",
     "gt": "경로에 '이탈리아'", "hit": lambda p, t: "이탈리아" in p},
    {"q": "아이슬란드 여행 때 오로라 찍은 사진 찾아줘", "type": "여행+장면",
     "gt": "경로에 '아이슬란드'", "hit": lambda p, t: "아이슬란드" in p},
    {"q": "몽골 여행에서 별이나 은하수 나온 사진 어디더라?", "type": "여행+장면",
     "gt": "경로에 mongolia/몽골", "hit": lambda p, t: "mongolia" in p or "몽골" in p},
    {"q": "제주도에서 해질녘 바다 보면서 찍은 일몰 사진 어디더라?", "type": "지명(국내)",
     "gt": "경로에 '제주'", "hit": lambda p, t: "제주" in p},
    {"q": "일산 호수공원에서 봄에 찍은 사진 찾아줘", "type": "지명(국내)",
     "gt": "경로에 '일산호수공원'", "hit": lambda p, t: "일산호수공원" in p},
    {"q": "결혼식 때 반지 클로즈업으로 찍은 사진 어디더라?", "type": "이벤트",
     "gt": "경로에 'wedding'", "hit": lambda p, t: "wedding" in p.lower()},
    {"q": "2013년에 찍은 밤 풍경 사진 보여줘", "type": "기간(연도)",
     "gt": "taken_at 2013년", "hit": lambda p, t: t.startswith("2013")},
]

eval_rows = []
for spec in EDDR_QUERIES:
    df = search(spec["q"], k=10)
    df["hit"] = [
        bool(spec["hit"](_nfc(p or ""), str(t or "")))
        for p, t in zip(df["image_path"], df["taken_at"])
    ]
    n_hit = int(df["hit"].sum())
    items = [{
        "path": r["image_path"],
        "label": f"{'✓' if r['hit'] else '✗'} #{r['rank']}  d={r['distance']:.2f} · {str(r['taken_at'])[:10]}\n"
                 + "\n".join(textwrap.wrap(textwrap.shorten(r["caption"], 100, placeholder="…"), 34)[:2]),
    } for _, r in df.head(5).iterrows()]
    show_photo_grid(items, title=f"'{spec['q']}'  —  GT({spec['gt']}) 적합 {n_hit}/10", cols=5, cell_in=2.4)
    eval_rows.append({"질의": spec["q"], "유형": spec["type"], "GT proxy": spec["gt"],
                      "hit@10": f"{n_hit}/10", "_n": n_hit})

eval_df = pd.DataFrame(eval_rows)
display(eval_df[["질의", "유형", "GT proxy", "hit@10"]])
print("\n유형별 평균 hit@10:")
display(eval_df.groupby("유형")["_n"].mean().round(1).rename("평균 적합 수(/10)").to_frame())
print("해석: GT는 '질문의 여행/기간 스코프' 기준 proxy다. 장면 자체는 맞지만 다른 여행이면 ✗로 집계되므로")
print("      semantic leg의 장면 검색력은 과소평가될 수 있다 — 그리드에서 ✗ 사진의 내용도 함께 볼 것.")

#### §D-5 육안 판정 — 2026-06-10, 사용자 직접 (증거 체인의 최종 레이어)

자동 GT(local 여행 폴더명 proxy)와 별개로, 위 그리드의 **표출 top-5**를 사람이 직접 판정한 결과.

| 질의 | 육안 판정 (top-5) | proxy 대비 해석 |
|---|---|---|
| 이탈리아 산악 | **5/5 전부 이탈리아** | proxy 3/10은 과소집계 — ✗는 iCloud 사본(cross-source 중복). **④ dedup 근거 육안 확정** |
| 아이슬란드 오로라 | 판정 불가 | 보유 오로라 사진이 전량 아이슬란드産 → 이 질의로는 지명 변별력 측정 불가(장면 검색력만 입증) |
| 몽골 은하수 | 2/5 (2·3번 국내, 4번 이탈리아) | 밤하늘 장면은 찾으나 장소 변별 실패 — 타지역·GPS 무 사진 혼입 |
| 제주 일몰 | 2/5 (1·3·5번 타지역) | proxy 0/10이었으나 2·4번은 실제 제주 — **폴더 proxy 블라인드스팟 실증**(photos_library 제주 사진은 proxy가 못 봄) |
| 일산 호수공원 | 3/5 (1·2번 오답) | proxy(2/10) 과소집계 — 그래도 지명 신호는 약함 |
| 결혼 반지 | 4/5 (5번 이탈리아, GPS 무) | top-3 정확(금반지 클로즈업) 유지 |
| 2013년 밤 풍경 | 타 연도 혼입 | 연도 축 부정확 확정 — 임베딩에 연도 신호 없음 |

**도출된 ⑦ 설계 입력(사용자 제안)**: 장소 질의 응답에서 **GPS 무 사진은 제거 또는 하단 정렬**, 혹은 **GPS 유무로 응답을 구획** → `TODO.md` ⑦에 등록(④ geocode와 연계).

> 종합: 폴더 proxy는 지명 축을 과소집계하지만(이탈리아 실질 5/5·제주 2/5), 방향은 동일 — 장면 검색은 강하고 장소·연도 변별은 캡션 단독으로 불충분. semantic + 메타(trip·geocode·날짜) 합성 필수.


## §E — 종합 리포트

모든 정량 게이트를 한 표로 모으고(§E-1), 과거 검증에서 보고됐던 설계-구현 불일치 3건이
문서·코드에 반영 유지되는지 매 실행 시 재검증한다(§E-2).

In [ ]:
# §E-1: 모든 검증 게이트 종합 (+ 데이터 품질 CHECK)
report = [
    ("§A 스키마 컬럼", "4 테이블 컬럼 완전 일치",
     "PASS" if schema_check["verdict"].eq("PASS").all() else "MISMATCH"),
    ("§A enum (source·status)", "실측 ⊆ 설계(SSoT)",
     "PASS" if enum_ok else "MISMATCH"),
    ("§A vector_id 형식", "100% 패턴 매칭",
     "PASS" if bool(emb["vid_ok"].all()) else "CHECK"),
    ("§B count 정합", f"SQLite={n_sql:,} == Chroma={n_chroma:,}",
     "PASS" if n_sql == n_chroma else "FAIL"),
    ("§B id 집합 diff", "양방향 0건",
     "PASS" if not only_sql and not only_chroma else "FAIL"),
    ("§B photo_id 무결성", "고아 0건",
     "PASS" if orphan_emb == 0 and orphan_cap == 0 else "FAIL"),
    ("§B 차원 일치", f"{vec_dim}-dim",
     "PASS" if len(_probe) == vec_dim else "FAIL"),
    ("§B-5 캡션 본문 전수 일치", f"불일치 {n_doc_mismatch}건 / {len(cap_df):,}",
     "PASS" if n_doc_mismatch == 0 else "FAIL"),
    ("§C-1 이미지 파일 실존", f"미존재 {n_missing_file}건 / {len(cap_ph):,}",
     "PASS" if n_missing_file == 0 else "FAIL"),
    ("§C-2 taken_at 파싱 가능", f"파싱 불가 {n_unparsable:,}건 (local EXIF 콜론 포맷)",
     "PASS" if n_unparsable == 0 else "CHECK"),
    ("§C-3 keywords 커버리지", f"{kw_cov * 100:.1f}% (bold+plain)",
     "PASS" if kw_cov >= 0.99 else "CHECK"),
    ("§D-3 self-retrieval", f"top-5 {top5_rate:.0%} (top-1 {top1_rate:.0%})",
     "PASS" if self_ok else "FAIL"),
]
report_df = pd.DataFrame(report, columns=["검증 항목", "기대/실측", "결과"])
display(report_df)

n_fail = int(report_df["결과"].isin(["FAIL", "MISMATCH"]).sum())
n_chk = int(report_df["결과"].eq("CHECK").sum())
if n_fail:
    print(f"종합: FAIL/MISMATCH {n_fail}건 — 위 표 확인")
elif n_chk:
    print(f"종합: 파이프라인 게이트 전부 PASS + 데이터 품질 CHECK {n_chk}건(후속 TODO) — 서비스 단계(⑦) 진행 가능")
else:
    print("종합: ALL PASS — 적재 데이터는 서비스 단계(⑦) 진행 근거 충족")

In [ ]:
# §E-2: 과거 불일치 3건 해소 확인 — 0beb879·2a52ad3 반영분을 문서·코드 원문에서 라이브 재검증
#   (이전 버전 §D-2가 보고했던 설계-구현 불일치. 문서가 되돌아가면 여기서 다시 잡힌다.)
resolved_checks = [
    ("indexing_status 'skipped_video' 설계 반영",
     ROOT / "docs/PLAN.md", "skipped_video"),
    ("Chroma distance metric L2 명시",
     ROOT / "CONTEXT.md", "Chroma 기본 L2"),
    ("repository.py kind 예시 caption_text 정정",
     ROOT / "src/eddr/db/repository.py", "예: ``caption_text``"),
]
rows = []
for name, path, needle in resolved_checks:
    found = path.exists() and needle in path.read_text(encoding="utf-8")
    rows.append({"항목": name, "근거 파일": str(path.relative_to(ROOT)),
                 "확인 문구": needle, "상태": "해소 확인" if found else "재발(문구 사라짐)"})
resolved_df = pd.DataFrame(rows)
display(resolved_df)
print("전부 해소:", "YES" if resolved_df["상태"].eq("해소 확인").all() else "NO — 문서 회귀 확인 필요")

---
### 결론

증거 사슬: **§A 스키마·enum 일치 → §B id·count·캡션 본문 전수 정합 → §C 파일 실존·필드 충전율·사진↔캡션 육안 대응 → §D 키워드/실사용 질의를 사진으로 확인 + self-retrieval 정량 게이트 + 실사용 질의 능력 지도**.

- **§E-1 파이프라인 게이트 전부 PASS**이고, §C-5·§D-2·§D-5 그리드의 의미 일치가 육안으로 확인되면,
  적재 데이터는 ⑦(LLM tool surface·검색 UX) 진행에 충분한 수준으로 구축된 것으로 판단한다.
- 과거 §D가 보고한 설계-구현 불일치 3건은 `0beb879`·`2a52ad3`로 해소 — §E-2가 매 실행 시 회귀를 감시한다.
- **§D-5 실사용 질의 함의**: 여행 고유 장면은 캡션 단독으로도 적중하지만, **지명·연도 축은 캡션에
  존재하지 않아 실패가 정상** — "어디더라?"형 질문의 완성은 ⑦에서 semantic + 메타(trip·geocode·날짜) 합성으로
  달성한다(후보 풀 선별 원칙과 동일한 결론, 이제 실측 수치 보유).
- 유의 사항 (이 노트북이 실측으로 적발한 후속 입력):
  1. **image_path 표기 혼재** (§C-1) — `google_takeout` 1,385건만 상대경로. 파일은 전부 실존하나
     CWD 의존 시 깨짐 → 표기 통일 또는 "루트 기준 resolve" 계약 명문화(TODO 등록됨).
  2. **taken_at EXIF 콜론 포맷** (§C-2) — local 806건이 `2018:04:10 18:38:51` 형태로 저장돼
     SQLite 날짜 함수에서 조용히 누락(나머지 924건은 원천 부재). ⑥ Trip·날짜 질의 전 로더 정규화 필요.
  3. **거리 임계값 정책 미정** (§D-4) — L2 절대값은 관련도 컷오프 불가. ⑦은 rank 기반으로 설계.
  4. **캡션 `Search keywords:` bold/plain 혼재** (§C-3) → ⑦ 파싱에서 두 패턴 모두 처리(TODO 반영됨).
  5. **RAW(rw2·orf·raf) 일부는 PIL 미리보기 불가** (§C-1) → 서비스 뷰어에서 변환 경로 필요.